# NBA Win Probability - Data Processing Pipeline

This notebook processes raw NBA play-by-play data (From Kaggle) and transforms it into a structured dataset suitable for machine learning. The pipeline includes data loading, cleaning, feature engineering, and preparation of the final training dataset.

## Import Libraries

Imports the Python libraries required for data manipulation, numerical operations, and file management throughout the data processing pipeline.

In [47]:
import pandas as pd
import numpy as np
import os
import glob

## Define Data Paths

This section establishes the locations of the raw NBA play-by-play files and the processed dataset output.

In [48]:
DATA_PATH = "../data/raw/nba_pbp"

OUTPUT_PATH = "../data/processed/ml_dataset.parquet"

## Load NBA Play-by-Play Data

Loads all available NBA play-by-play CSV files and combines them into a single dataframe. Each file is assigned a season label to preserve historical information.

In [49]:
all_files = sorted(
    glob.glob(os.path.join(DATA_PATH, "*.csv"))
)

dfs = []

for file in all_files:
    df_temp = pd.read_csv(file)

    season = os.path.basename(file).replace("pbp", "").replace(".csv", "")
    df_temp["season"] = int(season)

    dfs.append(df_temp)


df = pd.concat(
    dfs,
    ignore_index=True
)

print("Shape:", df.shape)
print("Games:", df["gameid"].nunique())

Shape: (18255730, 16)
Games: 37928


## Organize Game Events 

Sorts the play-by-play data by game, period, and clock time to ensure that events are processed in the correct order.

In [ ]:
df = df.sort_values(
    [
        "gameid",
        "period",
        "clock"
    ]
)

df = df.reset_index(drop=True)

## Fill Missing Score Information

This section fills missing score values using previous known scores within each game to maintain a continuous representation of the game state.

In [ ]:
df["h_pts"] = (df.groupby("gameid")["h_pts"].ffill())

df["a_pts"] = (df.groupby("gameid")["a_pts"].ffill())

## Score Differential Feature

Calculates the score difference between the home and away teams. Positive values indicate a home team lead, while negative values indicate an away team lead. Also fills missing score information with 0. 

In [ ]:
df["score_diff"] = (df["h_pts"] - df["a_pts"])

In [ ]:
df["score_diff"] = df["score_diff"].fillna(0)

## Convert Game Clock into Seconds

Converts the original play-by-play clock format into numerical seconds, allowing the model to understand the amount of time remaining in the quarter.

In [ ]:
def convert_clock(clock):

    clock = clock.replace("PT", "")

    minutes = clock.split("M")[0]

    seconds = (
        clock.split("M")[1].replace("S", "")
    )

    return int(minutes) * 60 + float(seconds)



df["quarter_seconds_elapsed"] = (
    df["clock"].apply(convert_clock)
)

MemoryError: Unable to allocate 139. MiB for an array with shape (18255730,) and data type int64

In [ ]:
df["quarter_seconds_remaining"] = (
    720 - df["quarter_seconds_elapsed"]
)

## Create Full Game Time Remaining Feature

Combines quarter time and overtime periods into a single game clock variable representing the total seconds remaining in the game.

In [ ]:
df["game_seconds_remaining"] = 0.0


regulation_mask = df["period"] <= 4


df.loc[
    regulation_mask,
    "game_seconds_remaining"
] = (

    (4 - df.loc[regulation_mask, "period"]) * 720

    +

    df.loc[
        regulation_mask,
        "quarter_seconds_remaining"
    ]
)

In [ ]:
overtime_mask = df["period"] > 4


df.loc[
    overtime_mask,
    "game_seconds_remaining"
] = (

    (df.loc[overtime_mask, "period"] - 5) * 300

    +

    df.loc[
        overtime_mask,
        "quarter_seconds_remaining"
    ]
)

## Create Momentum Feature

Creates a rolling momentum feature based on recent changes in score differential to capture short-term scoring trends during a game.

In [ ]:
df["score_diff_change"] = (
    df.groupby("gameid")["score_diff"]
    .diff()
)


df["score_diff_change"] = (
    df["score_diff_change"]
    .fillna(0)
)


df["momentum"] = (
    df.groupby("gameid")["score_diff_change"]
    .transform(
        lambda x:
        x.rolling(
            window=20,
            min_periods=1
        ).mean()
    )
)


df["momentum"] = df["momentum"].fillna(0)

## Create Nonlinear Score Differential Feature

Creates a squared score differential feature to allow the model to capture the diminishing impact of extremely large leads.

In [ ]:
df["score_diff_squared"] = (
    df["score_diff"] ** 2
)

## Generate Target Variable

Determines the final game outcome by comparing final home and away scores and creating the binary target variable used for supervised learning.

In [ ]:
final_scores = (
    df.groupby("gameid")
    .tail(1)
)


game_results = final_scores[
    [
        "gameid",
        "h_pts",
        "a_pts"
    ]
].copy()


game_results["home_win"] = (
    game_results["h_pts"] > game_results["a_pts"]
).astype(int)

## Attach Game Results to Each Game State

Adds the final game outcome to every play-by-play state, allowing the model to learn the relationship between in-game situations and final results.

In [ ]:
df = df.merge(
    game_results[
        [
            "gameid",
            "home_win"
        ]
    ],
    on="gameid",
    how="left"
)

## Create Machine Learning Dataset

Selects the final features required for model training and creates the processed dataset used in Notebook 2.

In [ ]:
ml_df = df[
    [
        "gameid",
        "season",
        "period",
        "game_seconds_remaining",
        "score_diff",
        "score_diff_squared",
        "momentum",
        "home_win"
    ]
].copy()

## Validate Processed Dataset

Checks the dataset shape, missing values, target distribution, and sample rows to ensure the data is ready for machine learning.

In [ ]:
print(ml_df.shape)

print(
    ml_df.isna().sum()
)

print(
    ml_df["home_win"].value_counts(normalize=True)
)

ml_df.head()

(18255730, 8)
gameid                    0
season                    0
period                    0
game_seconds_remaining    0
score_diff                0
score_diff_squared        0
momentum                  0
home_win                  0
dtype: int64
home_win
1    0.526295
0    0.473705
Name: proportion, dtype: float64


,gameid,season,period,game_seconds_remaining,score_diff,score_diff_squared,momentum,home_win
0,20000001,2001,1,2880.0,-3.0,9.0,0.00,0
1,20000001,2001,1,2879.9,0.0,0.0,1.50,0
2,20000001,2001,1,2879.8,0.0,0.0,1.00,0
3,20000001,2001,1,2856.7,0.0,0.0,0.75,0
4,20000001,2001,1,2853.7,0.0,0.0,0.60,0


## Export Processed Dataset

Saves the processed dataset as a parquet file, allowing Notebook 2 to efficiently load the data for model training.

In [ ]:
ml_df.to_parquet(
    OUTPUT_PATH,
    index=False
)

## Permanent Sample Dataset 

Creates a sample of the dataset with 1 million rows in order to reduce the strain on my computer. 

In [ ]:
ml_df_sample = ml_df.sample(
    n=2_000_000,
    random_state=42
)

ml_df_sample.to_parquet(
    "../data/processed/ml_dataset_sample.parquet",
    index=False
)

print(ml_df_sample.shape)

(2000000, 8)
